In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [2]:
class NN_Binary_Classifier:
    def __init__(self, X, M, epochs, L = 0.1):

        '''
        Takes parameters X: NxD Dataset, M: Hidden units,
        epochs: Number of training epochs, L: learning rate, 0.1 standard.
        Requires numpy imported
        '''

        self.M = M
        self.L = L
        self.epochs = epochs

        if X is not None:
            self.X = X
            self.N, self.D = X.shape
        else:
            print('Must enter valid dataset')

        #Step 1 Parameter inizialisastion
        self.W1 = np.random.randn(self.D, M) * 0.1
        self.b1 = np.zeros((1, M))
        self.W2 = np.random.randn(M, 1) * 0.1
        self.b2 = np.zeros((1, 1))


    #Step 2 Activation function
    def h(self, a):
        return 1 / (1 + np.exp(-a))
    
    
    #Step 3 Forward Pass
    def forward_propagation(self, X, W1, b1, W2, b2):
        A1 = X @ W1 + b1
        Z = self.h(A1)
        A2 = Z @ W2 + b2
        Y = self.h(A2)
        return A1, Z, A2, Y
    
    #Step 4 Compute error
    def compute_error(self, Y, T):
        e = 1e-8
        E = -np.mean(T * np.log(Y + e) + (1 - T) * np.log(1 - Y + e))
        return E
    
    #Step 5 Backwards pass
    def backward_propagation(self, X, T, W2, Z, Y):

        delta2 = Y - T
        delta1 = (delta2 @ W2.T) * Z * (1 - Z)

        grad_W1 = (X.T @ delta1) / self.N
        grad_b1 = np.sum(delta1, axis= 0, keepdims= True) / self.N
        grad_W2 = (Z.T @ delta2) / self.N
        grad_b2 = np.sum(delta2, axis = 0, keepdims= True) / self.N

        return grad_W1, grad_b1, grad_W2, grad_b2
    
    #Step 6 update parameters
    def update_parameters(self, W1, b1, W2, b2, grad_W1, grad_b1, grad_W2, grad_b2, L):
        W1 = W1 - L * grad_W1
        b1 = b1 - L * grad_b1
        W2 = W2 - L * grad_W2
        b2 = b2 - L * grad_b2
        return W1, b1, W2, b2
    
    def training(self, T, print_error_every = 5_000):

        '''
        Train the network in-place.
        Stores learned parameters in self.W1, self.b1, self.W2, self.b2.
        print_error_every controls how often loss is printed; set to 0 to disable.
        '''
        
        T = T.reshape(-1, 1)

        for epoch in range(self.epochs):
            A1, Z, A2, Y = self.forward_propagation(self.X, self.W1, self.b1, self.W2, self.b2)
            E = self.compute_error(Y, T)

            grad_W1, grad_b1, grad_W2, grad_b2 = self.backward_propagation(self.X, T, self.W2, Z, Y)
            self.W1, self.b1, self.W2, self.b2 = self.update_parameters(self.W1, self.b1, self.W2, self.b2,
                                                                        grad_W1, grad_b1, grad_W2, grad_b2,
                                                                        self.L)

            if print_error_every and epoch % print_error_every == 0 and epoch != 0:
                print(f"Epoch: {epoch}, Current Error: {E}")

        _, _, _, self.Y = self.forward_propagation(self.X, self.W1, self.b1, self.W2, self.b2)
        self.pred = (self.Y > 0.5).astype(int)
        self.T = T
        

    #Result evaluation
    def evaluate(self, X, T, name="Dataset"):
        T = T.reshape(-1, 1)

        _, _, _, Y = self.forward_propagation(X, self.W1, self.b1, self.W2, self.b2)
        pred = (Y > 0.5).astype(int)

        pred_flat = pred.reshape(-1)
        T_flat = T.reshape(-1)

        mistakes = np.sum(pred_flat != T_flat)
        acc = np.mean(pred_flat == T_flat)

        acc_class1 = np.mean(pred_flat[T_flat == 1] == 1)
        acc_class0 = np.mean(pred_flat[T_flat == 0] == 0)

        print()
        print('Dataset ->', name)
        print({"mistakes": mistakes,
                "accuracy_class_1": acc_class1,
                "accuracy_class_0": acc_class0,
                "accuracy": acc})


    def predict_prob(self, X):
        _, _, _, Y = self.forward_propagation(X, self.W1, self.b1, self.W2, self.b2)
        return Y 

    def predict(self, X):
        _, _, _, Y = self.forward_propagation(X, self.W1, self.b1, self.W2, self.b2)
        return (Y > 0.5).astype(int).reshape(-1)

Test network on standard XOR dataset to see how it handles nonlinearity
#XOR dataset from Kaggle at https://www.kaggle.com/datasets/bipinmaharjan/xor-dataset

In [32]:
'''Since this is a fictional random dataset there is no reason to split it'''

df = pd.read_csv('Xor_Dataset.csv')

X = np.array(df.iloc[:, 0:2])
T = np.array(df.iloc[:, 2]).reshape(-1, 1)

NN = NN_Binary_Classifier(X, 2, 10_000, L = 1)
NN.training(T, print_error_every= 1_000)
NN.evaluate(X, T, name= 'XOR')

Epoch: 1000, Current Error: 0.6930901541715674
Epoch: 2000, Current Error: 0.6930523977236782
Epoch: 3000, Current Error: 0.6928904304891215
Epoch: 4000, Current Error: 0.027784369321650956
Epoch: 5000, Current Error: 0.00716405465206939
Epoch: 6000, Current Error: 0.004063913076253238
Epoch: 7000, Current Error: 0.0028288497756886913
Epoch: 8000, Current Error: 0.0021671020778936166
Epoch: 9000, Current Error: 0.001755239329573837

Dataset -> XOR
{'mistakes': np.int64(0), 'accuracy_class_1': np.float64(1.0), 'accuracy_class_0': np.float64(1.0), 'accuracy': np.float64(1.0)}


Challenge the network more with a dataset on breastcancer from sklearn

In [3]:
breast_df = load_breast_cancer()
X = breast_df.data
T = breast_df.target

X_train, X_test, T_train, T_test = train_test_split(
    X, T, test_size=0.2, random_state=100)

'''We scale X because the features are on very different magnitudes,
which makes gradient-based training more stable and effective.'''

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Call neural network and print results
NN = NN_Binary_Classifier(X= X_train, M= 10, epochs= 100_000, L= 0.005)
NN.training(T_train, print_error_every= 10_000)
NN.evaluate(X_train, T_train, name= 'Training set')

#Call neural network and print results
NN.evaluate(X_test, T_test, name= 'Testing set')

Epoch: 10000, Current Error: 0.07823894574120437
Epoch: 20000, Current Error: 0.05869068183552674
Epoch: 30000, Current Error: 0.052274163438460294
Epoch: 40000, Current Error: 0.0489039703829127
Epoch: 50000, Current Error: 0.04671578040500926
Epoch: 60000, Current Error: 0.04506758472382062
Epoch: 70000, Current Error: 0.04367332254646122
Epoch: 80000, Current Error: 0.04240036200483832
Epoch: 90000, Current Error: 0.04119613838915014

Dataset -> Training set
{'mistakes': np.int64(3), 'accuracy_class_1': np.float64(1.0), 'accuracy_class_0': np.float64(0.9815950920245399), 'accuracy': np.float64(0.9934065934065934)}

Dataset -> Testing set
{'mistakes': np.int64(2), 'accuracy_class_1': np.float64(1.0), 'accuracy_class_0': np.float64(0.9591836734693877), 'accuracy': np.float64(0.9824561403508771)}
